# S3 - Calidad de datos y formatos analiticos particionados

**Actividad:** construir el notebook `03_calidad_datos_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), validando esquema, filtrando y ordenando resultados, tratando duplicados y nulos, y escribiendo una salida analitica particionada en Parquet — sobre el dataset real H&M ya usado en S2 (`customers.csv`, `articles.csv`).

**Proposito de la actividad:** dejar evidencia ejecutable de que dominas los controles de calidad de datos (esquema, filtrado, orden, duplicados, nulos) y el particionamiento de salidas analiticas — antes de avanzar a ML distribuido (S4).

Guia completa: `docs/sesiones/S03_Calidad_Datos_Particionamiento_Formatos_Analiticos.md`, seccion 3.

## 3.1 Preparar los datos de S3 y reanudar el entorno `lambda26`

**Producto del paso:** `customers.csv` y `articles.csv` disponibles en `pyspark/sesiones/s03-calidad-datos/data/`, entorno `lambda26` funcionando.

Ya descargaste estos dos archivos en S2 — no hace falta descargarlos de nuevo, solo copialos a la carpeta de esta sesion (desde tu maquina, no dentro del notebook):

```bash
cp lambda26/pyspark/sesiones/s02-fundamentos/data/customers.csv lambda26/pyspark/sesiones/s03-calidad-datos/data/
cp lambda26/pyspark/sesiones/s02-fundamentos/data/articles.csv lambda26/pyspark/sesiones/s03-calidad-datos/data/
```

`customers.csv` pesa ~207 MB — la copia tarda unos segundos, no es instantanea. **Espera a que termine antes de abrir Jupyter y correr el notebook**: si lees el archivo mientras todavia se esta copiando, Spark lee la foto parcial que existe en ese instante, sin ningun error. Confirma que la copia termino:

```bash
wc -l lambda26/pyspark/sesiones/s02-fundamentos/data/customers.csv
wc -l lambda26/pyspark/sesiones/s03-calidad-datos/data/customers.csv
```

Esta sesion no necesita `transactions.parquet` — el foco es esquema, filtrado, orden, duplicados y nulos sobre datos tabulares, no sobre transacciones.

## 3.2 Crear el notebook y la `SparkSession`

**Producto del paso:** notebook con una `SparkSession` activa.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion3-calidad-datos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

`spark.driver.memory` en `4g` desde el arranque — en S2 la JVM se cayo por quedarse en el default de 1g; aca se fija de una vez.

In [ ]:
ORIGEN_DATOS = "/opt/s03-calidad-datos/data"
ARTIFACTS = "/opt/s03-calidad-datos/artifacts"

## 3.3 Cargar `customers.csv` y validar el esquema

**Producto del paso:** `df_customers` cargado con esquema explicito, verificado contra lo esperado — control de calidad #1: esquema.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers,
)

df_customers.printSchema()

Confirma que el esquema real coincide con el documentado — 7 columnas, en el mismo orden y tipo:

In [ ]:
print(df_customers.columns)
df_customers.count()

## 3.4 Explorar nulos por columna

**Producto del paso:** conteo exacto de nulos por columna, con porcentaje sobre el total.

In [ ]:
from pyspark.sql.functions import col, count, when

total_filas = df_customers.count()

df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
]).show(vertical=True, truncate=False)

El mismo resultado, con porcentaje:

In [ ]:
nulos = df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
]).collect()[0].asDict()

for columna, cantidad in nulos.items():
    porcentaje = cantidad / total_filas * 100
    print(f"{columna}: {cantidad} nulos ({porcentaje:.1f}%)")

## 3.5 Filtrado de datos (`filter()`/`where()`)

**Producto del paso:** las dos sintaxis de `filter()` (SQL y booleana) aplicadas sobre datos reales, mas filtrado de nulos y de contenido de texto.

Expresion SQL como texto:

In [ ]:
df_customers.filter("age > 30").show(3)
df_customers.filter("club_member_status = 'ACTIVE'").show()
df_customers.filter("age BETWEEN 25 AND 35").show()
df_customers.filter("fashion_news_frequency IN ('Regularly', 'Monthly')").show()

La misma logica, con `col()`:

In [ ]:
from pyspark.sql.functions import col

df_customers.filter(col("age") > 30).show()
df_customers.filter(col("club_member_status") == "ACTIVE").show()
df_customers.filter((col("age") > 25) & (col("age") < 35)).show()
df_customers.filter(col("fashion_news_frequency").isin("Regularly", "Monthly")).show()

`where()` es el mismo metodo que `filter()`, con otro nombre:

In [ ]:
df_customers.where(col("Active") == 1).show()
df_customers.where("FN = 1").show()

Filtrar nulos de una columna especifica — a diferencia del conteo de 3.4, esto deja *ver* las filas, no solo contarlas:

In [ ]:
df_customers.filter(col("club_member_status").isNotNull()).show()
df_customers.filter(col("fashion_news_frequency").isNull()).show()

Filtrar por contenido de texto — util para validar formato:

In [ ]:
df_customers.filter(col("postal_code").startswith("28")).show()
df_customers.filter(col("postal_code").contains("56")).show()
df_customers.filter(col("postal_code").endswith("00")).show()

Filtrar tambien sirve para validar rangos — confirma si hay edades fuera de lo razonable:

In [ ]:
df_edad_invalida = df_customers.filter((col("age") < 0) | (col("age") > 100))
df_edad_invalida.count()

Si el conteo da 0, tambien es un control de calidad exitoso — no un resultado "vacio" sin valor.

## 3.6 Ordenar resultados (`orderBy()`/`sort()`)

**Producto del paso:** resultados ordenados por una, varias y por una expresion sobre una columna.

Por una sola columna, en formas equivalentes:

In [ ]:
df_customers.orderBy("age").show(3)
df_customers.orderBy(col("age")).show(3)
df_customers.orderBy(col("age").desc()).show(3)
df_customers.orderBy("age", ascending=False).show(3)

Por varias columnas, cada una con su propio sentido:

In [ ]:
df_customers.orderBy(col("club_member_status").asc(), col("age").desc()).show(3)
df_customers.orderBy(["club_member_status", "age"], ascending=[True, False]).show(3)

Por una expresion, no solo por el valor de la columna — aca, por la longitud del codigo postal:

In [ ]:
from pyspark.sql.functions import length

df_customers.orderBy(length(col("postal_code")).desc()).show()

`sort()` es el mismo metodo que `orderBy()`, con otro nombre:

In [ ]:
df_customers.sort("age").show(3)
df_customers.sort(col("age").desc()).show(3)

## 3.7 Tratamiento de duplicados

**Producto del paso:** duplicados identificados y/o tratados con varias tecnicas — control de calidad #2.

Eliminar duplicados completos o por columnas especificas:

In [ ]:
df_clean = df_customers.dropDuplicates()
df_clean = df_customers.dropDuplicates(["customer_id"])
df_clean = df_customers.dropDuplicates(["customer_id", "postal_code"])

`distinct()` es la forma corta de `dropDuplicates()` sin argumentos:

In [ ]:
df_clean = df_customers.distinct()

Identificar duplicados **sin** eliminarlos todavia:

In [ ]:
from pyspark.sql.functions import count

df_customers.groupBy("customer_id").count().filter("count > 1").show()

Confirma las cifras sobre el dataset completo — si coinciden con el total, no hay duplicados reales:

In [ ]:
total = df_customers.count()
sin_duplicados_fila_completa = df_customers.distinct().count()
sin_duplicados_por_id = df_customers.dropDuplicates(["customer_id"]).count()

print(f"Total: {total}, sin duplicar (fila completa): {sin_duplicados_fila_completa}, sin duplicar (por customer_id): {sin_duplicados_por_id}")

En una corrida real, las tres cifras dieron **1 371 980** — cero duplicados, en ninguna de las dos definiciones.

Marcar duplicados eligiendo cual fila conservar (aca, la de mayor `age` por `customer_id`):

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("customer_id").orderBy(col("age").desc())

df_ranked = df_customers.withColumn("row_num", row_number().over(window_spec))
df_clean = df_ranked.filter(col("row_num") == 1).drop("row_num")

**Contraste real con `articles.csv`** (S2, 3.10): `rdd.take(5)` sobre `detail_desc` trajo descripciones identicas repetidas. Eso **no** son duplicados de fila — cada `article_id` es distinto (variante de color/talla). Confirma cuantos `article_id` comparten la misma descripcion, sin tratarlos como error:

In [ ]:
df_articles = spark.read.csv(f"{ORIGEN_DATOS}/articles.csv", header=True, inferSchema=True)

duplicados_por_descripcion = (
    df_articles.filter(col("detail_desc").isNotNull())
    .groupBy("detail_desc")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)
duplicados_por_descripcion.show(5, truncate=False)

`filter(col("detail_desc").isNotNull())` va **antes** del `groupBy()`: sin el, agrupa todos los articulos sin descripcion bajo un mismo grupo `NULL` — que en una corrida real salio como el "valor mas repetido" (416 articulos), tapando los duplicados de contenido real.

Aca si corresponde `Window`+`row_number()` para elegir un representante por `product_code` (el producto base, sin variantes) — `article_id` identifica cada variante, `product_code` el producto:

In [ ]:
window_producto = Window.partitionBy("product_code").orderBy("article_id")

df_articles_un_por_producto = (
    df_articles
    .withColumn("fila", row_number().over(window_producto))
    .filter(col("fila") == 1)
    .drop("fila")
)

print(f"Filas originales: {df_articles.count()}, un representante por product_code: {df_articles_un_por_producto.count()}")

En una corrida real, la reduccion fue de **105 542 filas a 47 224 representantes** — mas de la mitad de `articles.csv` son variantes de un producto ya representado por otra fila.

## 3.8 Tratar nulos con `.na.fill()` y `.na.drop()`

**Producto del paso:** `df_customers_valido`, el dataset final — con nulos tratados columna por columna, cada decision con un criterio documentado — control de calidad #3.

`FN`/`Active` son columnas de tipo "bandera"; un nulo ahi significa "la bandera no se activo" — se rellenan con `0`. `fillna()` es un alias exacto de `.na.fill()`:

In [ ]:
df_fill2 = df_customers.fillna({"FN": 0, "Active": 0})

`.na.fill()` puede rellenar cualquier columna, con cualquier tipo de valor — incluida `age`, con `0`. Pero que la sintaxis lo permita no lo hace buena idea: un cliente de "0 anos" es un dato **falso** que se ve como valido, peor que dejarlo nulo. Por eso la version que este notebook aplica de verdad **no** rellena `age`:

In [ ]:
df_customers_limpio = df_customers.na.fill({
    "FN": 0,
    "Active": 0,
    "fashion_news_frequency": "NONE",
    "club_member_status": "UNKNOWN",
})

`.na.drop()` sin argumentos elimina toda fila con **cualquier** nulo, en cualquier columna — sobre este dataset (FN/Active ~65% nulos), descartaria la enorme mayoria de las filas. Pruebalo para ver la magnitud, pero no lo uses como version final:

In [ ]:
df_customers.na.drop().count()

`customer_id` es la columna critica — corresponde `.na.drop(subset=[...])`, apuntando solo a esa columna:

In [ ]:
df_customers_valido = df_customers_limpio.na.drop(subset=["customer_id"])

print(f"Filas antes: {df_customers.count()}, despues de na.drop(subset=['customer_id']): {df_customers_valido.count()}")

En una corrida real, ambos numeros dieron **1 371 980** — `customer_id` nunca llega nulo en este dataset.

`df_customers_valido` se reutiliza en los pasos que siguen (3.9-3.11), varios con su propio `.count()` — cache() guarda el resultado la primera vez que una accion lo dispara:

In [ ]:
df_customers_valido = df_customers_valido.cache()

Si el DataFrame fuera mas grande de lo que la memoria disponible aguanta, `persist(StorageLevel.MEMORY_AND_DISK)` es la version con mas control — cae a disco en vez de fallar:

In [ ]:
from pyspark.storagelevel import StorageLevel

df_customers_valido.persist(StorageLevel.MEMORY_AND_DISK)

## 3.9 Escritura en multiples formatos

**Producto del paso:** el mismo resultado guardado en tres formatos distintos, con `.write.format()` — sobre una muestra chica, no el dataset completo, solo para ver la sintaxis.

In [ ]:
muestra = df_edad_invalida.limit(100)  # resultado (vacio o no) de la validacion de 3.5

muestra.write.format("csv").option("header", True).mode("overwrite").save(f"{ARTIFACTS}/muestra_csv")
muestra.write.format("json").mode("overwrite").save(f"{ARTIFACTS}/muestra_json")
muestra.write.format("parquet").mode("overwrite").save(f"{ARTIFACTS}/muestra_parquet")

## 3.10 Escritura particionada en Parquet

**Producto del paso:** salida analitica particionada por `club_member_status`, lista para BI/ML.

`repartition(4)` antes de escribir controla cuantos archivos caen dentro de **cada** carpeta de particion — sin esto, se hereda el numero de particiones de la lectura original:

In [ ]:
(
    df_customers_valido
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("club_member_status")
    .save(f"{ARTIFACTS}/customers_particionado")
)

`partitionBy("club_member_status")` crea una subcarpeta por cada valor distinto de esa columna. Verifica la estructura real:

In [ ]:
import os

for carpeta in sorted(os.listdir(f"{ARTIFACTS}/customers_particionado")):
    print(carpeta)

Contraste directo: si en vez de `repartition(4)` usas `coalesce(1)`, obtenes un solo archivo por carpeta de particion en vez de cuatro:

In [ ]:
(
    df_customers_valido
    .coalesce(1)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("club_member_status")
    .save(f"{ARTIFACTS}/customers_particionado_un_archivo")
)

## 3.11 Leer de vuelta y verificar el particionamiento

**Producto del paso:** confirmacion de que la salida particionada se lee correctamente y que el particionamiento si se aprovecha en consultas.

In [ ]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/customers_particionado")
df_verificacion.printSchema()
df_verificacion.count()

`club_member_status` reaparece en el esquema aunque no esta dentro de los archivos Parquet fisicos — Spark lo reconstruye a partir del nombre de la carpeta.

Filtra por la columna particionada y revisa el plan — deberias ver `PartitionFilters`, no solo `PushedFilters` (el que ya viste en S2, 3.6):

In [ ]:
df_verificacion.filter(col("club_member_status") == "ACTIVE").explain(True)

Ya terminaste de reutilizar `df_customers_valido` — libera la memoria que ocupaba cacheado:

In [ ]:
df_customers_valido.unpersist()

## 3.12 Documentar hallazgos y responder preguntas de reflexion

**Producto del paso:** notebook documentado con celdas markdown explicando cada resultado.

Agrega celdas markdown breves debajo de cada bloque de codigo explicando que hiciste y que observaste — es la base directa de la evidencia tecnica para 4.3.1.

**Reflexion tecnica breve** (5 a 8 lineas): ¿que diferencia encontraste entre dropDuplicates() y Window+row_number() al aplicarlos sobre articles.csv? ¿que columnas rellenaste con na.fill() y cuales no, y por que? ¿que diferencia notaste entre PushedFilters (S2) y PartitionFilters (S3) en el plan de ejecucion?

_(Responde aqui)_